# الفصل 10: الانتباه الذاتي متعدد الرؤوس

> إذا كان الفصل التاسع قد أرانا كيف "أضفنا" Attention فوق RNN، فهذا الفصل يُريك الفكرة الثورية الحقيقية: ماذا لو تخلّصنا من RNN تمامًا، وجعلنا كل كلمة "تنتبه" مباشرة لكل كلمة أخرى في نفس الجملة؟ هذه هي فكرة...

- **الجزء:** 3 — المحوّلات الأساسية
- **الفصل:** 10
- **المستوى:** متقدم
- **زمن القراءة:** 20 دقيقة
- **المتطلبات:** الفصل 9: من التكرار إلى الانتباه: آلية Attention التاريخية
- **الوسوم:** Self-Attention، Multi-Head، QKV، Masks

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch10-self-attention.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-03-transformers%2Fch10-self-attention.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-03-transformers/ch10-self-attention.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch10-self-attention.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch10-self-attention.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch10-self-attention.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** numpy, torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("numpy", "numpy"),
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch10-self-attention.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-03-transformers%2Fch10-self-attention.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-03-transformers/ch10-self-attention.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch10-self-attention.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch10-self-attention.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch10-self-attention.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> إذا كان الفصل التاسع قد أرانا كيف "أضفنا" Attention فوق RNN، فهذا الفصل يُريك الفكرة الثورية الحقيقية: ماذا لو تخلّصنا من RNN تمامًا، وجعلنا كل كلمة "تنتبه" مباشرة لكل كلمة أخرى في نفس الجملة؟ هذه هي فكرة...

> **بيانات الفصل**
>
> - **الجزء:** 3 — المحوّلات الأساسية
> - **المستوى:** متقدم
> - **زمن القراءة:** 20 دقيقة
> - **المتطلبات:** الفصل 9: من التكرار إلى الانتباه: آلية Attention التاريخية


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم Self-Attention
- فهم Multi-Head
- فهم QKV


## 1. لماذا يهمّك هذا الموضوع؟

إذا كان الفصل التاسع قد أرانا كيف "أضفنا" Attention فوق RNN، فهذا الفصل يُريك الفكرة الثورية الحقيقية: **ماذا لو تخلّصنا من RNN تمامًا، وجعلنا كل كلمة "تنتبه" مباشرة لكل كلمة أخرى في نفس الجملة؟** هذه هي فكرة **Self-Attention**، قلب معمارية Transformer، والآلية التي جعلت التدريب المتوازي الكامل على معالجات GPU ممكنًا، وأتاحت بناء نماذج بمليارات المعاملات.

فهم Self-Attention وMulti-Head Attention بعمق — رياضيًا وبرمجيًا — هو **أهم مهارة تقنية واحدة** يمكن أن تمتلكها إذا كنت تعمل في مجال معالجة اللغة الطبيعية الحديثة، لأن كل نموذج لغوي كبير (GPT، BERT، LLaMA، Claude) مبني على تكديس آلاف من طبقات هذه الآلية بالضبط.

---


## 2. الخريطة الذهنية


```text
Self-Attention: "كل كلمة تسأل كل كلمة أخرى: كم أنتِ مهمة لي؟"

لكل كلمة في الجملة نشتق ثلاثة متجهات من تضمينها الأصلي:
│
├── Query (Q): "ماذا أبحث عنه؟" (استعلام الكلمة الحالية)
├── Key (K):   "ماذا أُقدّم كمحتوى؟" (مفتاح كل كلمة في الجملة)
└── Value (V): "ما المعلومة الفعلية التي أحملها؟"

خطوات الحساب (Scaled Dot-Product Attention):
│
1. QK^T  → درجات تشابه بين كل زوج كلمات (مصفوفة n×n)
2. / √d_k → تحجيم لمنع انفجار القيم مع زيادة الأبعاد
3. + Mask (اختياري) → منع النظر لمواضع معينة (مستقبل/padding)
4. softmax → تحويل الدرجات لأوزان احتمالية تجمع لـ 1 بكل صف
5. × V    → تجميع موزون لمتجهات القيمة → تمثيل جديد لكل كلمة

Multi-Head Attention: لماذا رأس واحد لا يكفي؟
│
├── رأس واحد يتعلم "نوعًا واحدًا" من العلاقات
│   (مثال: فقط العلاقات النحوية، أو فقط المرجعية الضميرية)
│
├── نُقسّم d_model إلى h رؤوس متوازية، كل رأس بأبعاد d_k = d_model/h
│   كل رأس يتعلم Q,K,V خاصة به → "منظور" مختلف للعلاقات
│
└── ندمج مخرجات كل الرؤوس ونمرّرها عبر طبقة خطية W_O
    → تمثيل غني متعدد الأوجه لكل كلمة
```


---


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

تخيّل حفل كوكتيل فيه عشرة أشخاص يتحدثون في نفس الوقت. كل شخص (كلمة) يحمل:
- **بطاقة تعريف (Query)**: "أنا أبحث عن معلومات عن كذا"
- **لافتة صدر (Key)**: "أنا أتحدث عن كذا"
- **محتوى فعلي (Value)**: المعلومة التي يحملها فعلًا

عندما تريد كلمة "it" مثلًا أن تفهم ما تشير إليه في جملة "The cat sat on the mat because **it** was tired"، فإن Query الخاص بـ"it" "يبحث" بين كل الـ Keys في الجملة، فيجد أن Key كلمة "cat" يُطابق بقوة (لأن "cat" هو الفاعل الحيّ المنطقي الذي "يتعب")، فتحصل كلمة "cat" على وزن انتباه مرتفع، ويُسحب الـ Value الخاص بها بقوة إلى التمثيل الجديد لكلمة "it".

### 🟡 طبقة الآلية: الصيغة الرياضية الكاملة

لنفترض تسلسل مدخل بطول $n$ ممثّلًا بمصفوفة تضمينات $X \in \mathbb{R}^{n \times d_{model}}$. نشتق ثلاث مصفوفات بضرب $X$ في مصفوفات أوزان قابلة للتعلم:

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$

حيث $W^Q, W^K, W^V \in \mathbb{R}^{d_{model} \times d_k}$.

**Scaled Dot-Product Attention**:

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**لماذا نقسم على $\sqrt{d_k}$؟** هذا سؤال مفصلي شائع في المقابلات. بافتراض أن عناصر $Q$ و$K$ متغيرات عشوائية مستقلة بمتوسط صفر وتباين 1، فإن حاصل الضرب النقطي $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ له تباين يساوي $d_k$ (مجموع d_k حدود، كل منها تباينه 1 تقريبًا). أي أن **تباين الدرجات ينمو خطيًا مع بُعد المتجهات**. عندما يكون $d_k$ كبيرًا (مثلًا 64 أو 128)، تصبح بعض الدرجات كبيرة جدًا موجبة أو سالبة، مما يدفع softmax إلى مناطق مشبعة (saturated regions) حيث التدرج يقترب من الصفر تقريبًا لكل العناصر عدا واحد — وهذا يُبطئ التعلم بشدة أو يُوقفه (vanishing gradients في softmax). القسمة على $\sqrt{d_k}$ تُعيد التباين إلى 1 تقريبًا، فتُبقي توزيع softmax في منطقة متوازنة ذات تدرجات صحية.

**Masking**: نحتاج لإخفاء بعض المواضع في حالتين:
1. **Causal Mask** (في فك التشفير/decoder): تمنع كل موضع من النظر إلى المواضع **المستقبلية** (لضمان الخاصية الذاتية الانحدارية autoregressive):
$$\text{mask}_{i,j} = \begin{cases} 0 & j \leq i \\ -\infty & j > i \end{cases}$$
2. **Padding Mask**: تمنع الانتباه لرموز الحشو الفارغة التي أُضيفت لتوحيد أطوال الدفعة.

**Multi-Head Attention**:

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

حيث $d_k = d_v = d_{model}/h$ عادة (مثلًا $d_{model}=512, h=8 \rightarrow d_k=64$).

### 🔴 طبقة التعمّق: الكود الكامل


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import math

# ============================================================
# 1. Scaled Dot-Product Attention من الصفر بـ NumPy
#    (لفهم الآلية الرياضية حرفيًا دون أي تجريد)
# ============================================================
def softmax_numpy(x, axis=-1):
    """Softmax مستقرة عدديًا."""
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def scaled_dot_product_attention_numpy(Q, K, V, mask=None):
    """
    Q: (n, d_k)   K: (n, d_k)   V: (n, d_v)
    mask: (n, n) اختياري، صفر حيث نمنع الانتباه

    العائد: (output, attn_weights)
    """
    d_k = Q.shape[-1]

    # QK^T  → (n, n)
    scores = Q @ K.T

    # التحجيم: القسمة على جذر d_k
    scores = scores / np.sqrt(d_k)

    # الإخفاء (إن وُجد)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)

    # softmax على كل صف (لكل استعلام، توزيع احتمالي على كل المفاتيح)
    attn_weights = softmax_numpy(scores, axis=-1)

    # التجميع الموزون للقيم
    output = attn_weights @ V

    return output, attn_weights

# ============================================================
# 2. Multi-Head Attention كامل — PyTorch (جاهز للإنتاج)
# ============================================================
class MultiHeadAttention(nn.Module):
    """
    تطبيق كامل لـ Multi-Head Attention كما في ورقة
    "Attention Is All You Need" (Vaswani et al., 2017).
    """
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model يجب أن يُقسَم على num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # إسقاطات خطية موحدة لكل الرؤوس دفعة واحدة (أكفأ حسابيًا من رؤوس منفصلة)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """ (batch, seq_len, d_model) → (batch, num_heads, seq_len, d_k) """
        batch_size, seq_len, _ = x.shape
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def combine_heads(self, x: torch.Tensor) -> torch.Tensor:
        """ (batch, num_heads, seq_len, d_k) → (batch, seq_len, d_model) """
        batch_size, _, seq_len, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(batch_size, seq_len, self.d_model)

    def forward(self, query, key, value, mask=None):
        """
        query, key, value: (batch, seq_len, d_model)
        mask: (batch, 1, seq_len_q, seq_len_k) أو قابل للبث (broadcastable)
        """
        Q = self.split_heads(self.W_q(query))   # (batch, h, seq_len, d_k)
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))

        # QK^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # scores: (batch, h, seq_len_q, seq_len_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-1e9'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, V)      # (batch, h, seq_len_q, d_k)
        context = self.combine_heads(context)         # (batch, seq_len_q, d_model)

        output = self.W_o(context)
        return output, attn_weights

# ============================================================
# 3. أقنعة جاهزة (Masks)
# ============================================================
def create_causal_mask(seq_len: int) -> torch.Tensor:
    """قناع سببي (ذاتي الانحدار) يمنع النظر للمستقبل."""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)

def create_padding_mask(token_ids: torch.Tensor, pad_token_id: int = 0) -> torch.Tensor:
    """قناع يخفي رموز الحشو."""
    mask = (token_ids != pad_token_id).unsqueeze(1).unsqueeze(2)
    return mask  # (batch, 1, 1, seq_len)


---


## 4. مثال محلول خطوة بخطوة

لنأخذ جملة عربية مبسّطة من كلمتين: **"الملك حكيم"**، بتمثيل تضمينات ثنائية البعد (d_model=2 للتبسيط، ومصفوفات وزن بسيطة $d_k=2$):

$$x_1 = [1, 0] \ (\text{"الملك"}), \qquad x_2 = [0, 1] \ (\text{"حكيم"})$$

لنفترض مصفوفات الأوزان (بسيطة لأغراض الحساب اليدوي):

$$W^Q = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}, \quad W^K = \begin{bmatrix} 1 & 1 \\ 0 & 1 \end{bmatrix}, \quad W^V = \begin{bmatrix} 1 & 0 \\ 1 & 1 \end{bmatrix}$$

**حساب Q، K، V**:

$$Q = XW^Q = \begin{bmatrix}1&0\\0&1\end{bmatrix}, \quad K = XW^K = \begin{bmatrix}1&1\\0&1\end{bmatrix}, \quad V = XW^V = \begin{bmatrix}1&0\\1&1\end{bmatrix}$$

**حساب $QK^T$**:

$$QK^T = \begin{bmatrix}1&0\\0&1\end{bmatrix}\begin{bmatrix}1&0\\1&1\end{bmatrix} = \begin{bmatrix}1&0\\1&1\end{bmatrix}$$

**القسمة على $\sqrt{d_k}=\sqrt{2}\approx1.414$**:

$$\text{scores} = \begin{bmatrix}0.707&0\\0.707&0.707\end{bmatrix}$$

**تطبيق softmax على كل صف**:

الصف الأول (استعلام "الملك"): $\text{softmax}([0.707, 0]) = \left[\frac{e^{0.707}}{e^{0.707}+e^0}, \frac{e^0}{e^{0.707}+e^0}\right] \approx [0.669, 0.331]$

الصف الثاني (استعلام "حكيم"): $\text{softmax}([0.707, 0.707]) = [0.5, 0.5]$ (قيمتان متساويتان → توزيع متساوٍ)

**مصفوفة الانتباه النهائية**:

$$\alpha = \begin{bmatrix}0.669 & 0.331 \\ 0.5 & 0.5\end{bmatrix}$$

**حساب المخرج النهائي ($\alpha \times V$)**:

لـ"الملك": $0.669 \times [1,0] + 0.331 \times [1,1] = [1.0, 0.331]$

لـ"حكيم": $0.5 \times [1,0] + 0.5 \times [1,1] = [1.0, 0.5]$

**التفسير**: تمثيل "الملك" بعد Self-Attention أصبح $[1.0, 0.331]$ — وهو مزيج يعتمد بشكل أكبر على قيمة "الملك" نفسه (وزن 0.669) مع إسهام أقل من "حكيم" (0.331). هذا منطقي حدسيًا: الكلمة "تنتبه" لنفسها بقوة أكبر عادة، مع استيعاب بعض السياق من الكلمات المجاورة.

---


## 5. نصائح وحيل

> 💡 **تحقّق دائمًا أن مجموع كل صف في مصفوفة الانتباه = 1**: هذا اختبار صحة أساسي (sanity check) بعد تطبيق softmax — إذا لم يتحقق، فهناك خطأ في المحور (axis) المستخدم.

> 💡 **استخدم `-1e9` بدلاً من `-inf` الحرفي عند الإخفاء**: `float('-inf')` قد يُنتج `NaN` عند بعض عمليات الـ backward في بعض إصدارات PyTorch إذا كان الصف بأكمله مُخفى؛ `-1e9` آمن عدديًا ويُعطي تأثيرًا مطابقًا عمليًا بعد softmax.

> 💡 **راقب أوزان كل رأس على حدة أثناء التحليل**: أظهرت أبحاث (مثل "What Does BERT Look At?") أن رؤوسًا معينة تتخصص تلقائيًا في علاقات نحوية محددة (مثل الفاعل-الفعل) دون أي إشراف صريح — افحص هذا بصريًا لفهم ما تعلمه نموذجك.

> 💡 **Multi-Head أرخص مما يبدو**: رغم استخدام h رؤوس، فإن كل رأس بأبعاد $d_k = d_{model}/h$، لذا فإن التعقيد الحسابي الكلي لـ Multi-Head Attention مماثل تقريبًا لرأس واحد بأبعاد $d_{model}$ كاملة — المكسب هو التنوع في التمثيل وليس تكلفة إضافية كبيرة.

---


## 6. أخطاء شائعة ومصائد

> ⚠️ **نسيان القسمة على $\sqrt{d_k}$**: يؤدي إلى softmax شديدة التشبّع (قيم قريبة من 0 أو 1 فقط) وتدرجات شبه معدومة، مما يُبطئ أو يُفشل التدريب تمامًا في الشبكات العميقة.

> ⚠️ **الخلط بين `mask==0` و`mask==1` في الإخفاء**: تأكد من اتفاقية القناع المستخدمة (1=مسموح، 0=ممنوع هي الأشيع) وطبّقها بثبات في كل الكود، فالخطأ هنا يعني أن النموذج "يرى" ما يجب إخفاؤه والعكس.

> ⚠️ **تطبيق Causal Mask في Encoder عن طريق الخطأ**: الـ Encoder (كما في BERT) يجب أن يرى كل التسلسل بدون قيود سببية (bidirectional) — تطبيق قناع سببي هناك يُحوّل النموذج فعليًا إلى شبه-decoder ويُضعف الأداء بشدة.

> ⚠️ **الخلط بين `d_model` و`d_k` في أبعاد المصفوفات**: خطأ شائع عند البرمجة اليدوية هو نسيان تقسيم `d_model` على `num_heads` بشكل صحيح قبل حساب `attention scores`، مما يُنتج أخطاء أبعاد (shape mismatch) يصعب تتبعها أحيانًا.

---


## 7. جدول مقارنة

| الخاصية | Single-Head Attention | Multi-Head Attention |
|---|---|---|
| **عدد "منظورات" العلاقة** | واحد فقط | h منظورات متوازية |
| **القدرة التمثيلية** | محدودة، قد يُفوّت أنواعًا من العلاقات | غنية، تلتقط أنواعًا متعددة (نحوية، دلالية، مرجعية) |
| **البعد لكل رأس** | $d_{model}$ كاملة | $d_{model}/h$ لكل رأس |
| **التكلفة الحسابية الكلية** | أساسية | مماثلة تقريبًا (بفضل تقسيم الأبعاد) |
| **القابلية للتفسير** | أبسط للتحليل | أصعب قليلًا (لكن يمكن تحليل كل رأس على حدة) |
| **الاستخدام الفعلي** | نادر في الممارسة الحديثة | المعيار الافتراضي في كل Transformer حديث |

---


## 8. ملخص أفضل الممارسات

- ✅ تأكد دائمًا من القسمة على $\sqrt{d_k}$ لتثبيت تدرجات softmax.
- ✅ استخدم إسقاطات خطية موحدة (`d_model → d_model`) ثم أعد التشكيل (reshape) لتقسيم الرؤوس، بدلاً من رؤوس منفصلة فعليًا — أكفأ حسابيًا على GPU.
- ✅ فرّق بوضوح بين Causal Mask (للـ decoder) وPadding Mask (لكليهما)، وتأكد من دمجهما بشكل صحيح عند الحاجة (عبر الضرب المنطقي أو الجمع).
- ✅ استخدم `num_heads` يقسم `d_model` تمامًا بلا باقٍ (قيد تصميمي إلزامي).
- ✅ فعّل Dropout على أوزان الانتباه (وليس فقط على المخرجات) لتحسين التعميم.

---


## 9. أسئلة مقابلات

**س1: اشرح الفرق المفاهيمي بين Query وKey وValue في Self-Attention.**

**الإجابة النموذجية**: جميعها إسقاطات خطية مختلفة لنفس متجه التضمين الأصلي، لكنها تلعب أدوارًا مختلفة وظيفيًا: الـ **Query** يمثّل "ما الذي تبحث عنه" الكلمة الحالية من معلومات في بقية التسلسل؛ الـ **Key** يمثّل "ما الذي تُعلن الكلمة أنها تحتوي عليه" (نوع المعلومة المتاحة)؛ والـ **Value** هو المحتوى الفعلي الذي يُسحَب ويُدمَج إذا كان التوافق بين Query وKey عاليًا. المقارنة الشائعة هي بمحرك بحث: Query هو ما تكتبه في شريط البحث، Key هي الكلمات المفتاحية المرتبطة بكل صفحة ويب، وValue هو المحتوى الفعلي للصفحة الذي تحصل عليه إذا تطابق بحثك مع كلماتها المفتاحية.

**س2: لماذا نحتاج Multi-Head Attention بدلاً من الاكتفاء برأس واحد بنفس الأبعاد الكلية؟**

**الإجابة النموذجية**: رأس واحد، مهما كبر بُعده، يُنتج توزيع انتباه واحدًا فقط لكل زوج كلمات — وهذا يُجبر النموذج على ضغط كل أنواع العلاقات الممكنة (النحوية، الدلالية، المرجعية، الموضعية...) في نمط واحد مشترك، مما يُضعف القدرة التمثيلية. بتقسيم الحساب إلى h رؤوس متوازية، كل رأس بمساحة إسقاط مستقلة، يستطيع النموذج تعلم أنماط انتباه متخصصة ومختلفة تمامًا في آن واحد (رأس يركّز على علاقات الفاعل-الفعل، آخر على المرجعية الضميرية، ثالث على القرب الموضعي...)، ثم تُدمَج كل هذه "الآراء" معًا عبر طبقة خطية $W^O$ في تمثيل نهائي أغنى بكثير.

**س3: ما الفرق بين Self-Attention وCross-Attention؟**

**الإجابة النموذجية**: في Self-Attention، تُشتق Q وK وV جميعها من **نفس التسلسل** — أي أن كل عنصر في تسلسل معين ينتبه لعناصر من نفس التسلسل (هذا ما يحدث داخل طبقات الـ Encoder وداخل الجزء الذاتي من طبقات الـ Decoder). أما في Cross-Attention (المستخدم في معماريات Encoder-Decoder مثل الترجمة الآلية أو T5)، فإن Q تُشتق من تسلسل واحد (عادة مخرجات الـ Decoder حتى الآن) بينما K وV تُشتقان من تسلسل **مختلف** تمامًا (مخرجات الـ Encoder الكاملة) — وهذا يُمكّن المفكك من "الاستعلام" عن المعلومات المصدر أثناء التوليد، تمامًا كما كانت تفعل Bahdanau/Luong Attention لكن بصياغة رياضية موحدة وقابلة للتوازي الكامل.

**س4: ما تعقيد Self-Attention الحسابي والذاكري من حيث طول التسلسل n وبُعد النموذج d؟**

**الإجابة النموذجية**: التعقيد الزمني هو $O(n^2 \cdot d)$ والتعقيد الذاكري لمصفوفة الانتباه هو $O(n^2)$، لأننا نحسب درجة توافق بين **كل زوج** من المواضع الـ n — أي مصفوفة $n \times n$. هذا يعني أن مضاعفة طول التسلسل تُربّع تقريبًا الكلفة الحسابية والذاكرية، وهو السبب الرئيسي وراء صعوبة تدريب نماذج Transformer على سياقات طويلة جدًا (آلاف أو عشرات آلاف الرموز) دون تقنيات تحسين خاصة مثل Flash Attention أو آليات الانتباه المتفرق (sparse attention) — وهو الموضوع الذي سنتعمق فيه في الجزء الرابع من الكتاب.

---


## 10. تمرين عملي

1. نفّذ `scaled_dot_product_attention_numpy` وطبّقها على جملة من 4 كلمات عربية بتضمينات عشوائية ثلاثية الأبعاد، وتحقّق يدويًا من أن مجموع كل صف في مصفوفة الانتباه الناتجة يساوي 1.
2. عدّل الكود لإضافة Causal Mask، وتحقق أن الكلمة الأولى تنتبه فقط لنفسها (وزن = 1.0 بالكامل).
3. وسّع المثال إلى `MultiHeadAttention` بـ PyTorch بأربعة رؤوس، واطبع شكل (`shape`) كل من Q وK وV بعد `split_heads` للتأكد من فهمك لعملية إعادة التشكيل.
4. قارن بصريًا (heatmap) بين خريطة انتباه رأس واحد عشوائي التهيئة وخريطة انتباه بعد تدريب قصير جدًا على مهمة تصنيف بسيطة — لاحظ كيف تتغير الأنماط.

---


## 11. مراجع للتعمّق

- Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.
- Clark, K., et al. (2019). *What Does BERT Look At? An Analysis of BERT's Attention*. BlackboxNLP.
- Alammar, J. *The Illustrated Transformer* (مقالة بصرية تفاعلية ممتازة).


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — المحوّلات الأساسية / الفصل 10: الانتباه الذاتي متعدد الرؤوس.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
